In [14]:
import os, zipfile, urllib.request
import pandas as pd
import numpy as np
from datetime import datetime

for folder in ["bronze", "silver", "gold", "raw"]:
    os.makedirs(folder, exist_ok=True)

url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"

req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp, open("raw/book-crossings.zip", "wb") as f:
    f.write(resp.read())

with zipfile.ZipFile("raw/book-crossings.zip") as z:
    z.extractall("raw")
    print(z.namelist())

['BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv']


In [15]:
def read_raw(path):
    return pd.read_csv(path, sep=";", encoding="latin-1",
                       on_bad_lines="skip", low_memory=False,
                       dtype=str)

users   = read_raw("raw/BX-Users.csv")
books   = read_raw("raw/BX-Books.csv")
ratings = read_raw("raw/BX-Book-Ratings.csv")

ts = datetime.now()
for df in (users, books, ratings):
    df["ingestion_timestamp"] = ts

users.to_parquet("bronze/users.parquet", index=False)
books.to_parquet("bronze/books.parquet", index=False)
ratings.to_parquet("bronze/ratings.parquet", index=False)

In [16]:
users   = pd.read_parquet("bronze/users.parquet")
books   = pd.read_parquet("bronze/books.parquet")
ratings = pd.read_parquet("bronze/ratings.parquet")

# 2.1 Shape, dtypes, missing values
for name, df in [("users", users), ("books", books), ("ratings", ratings)]:
    print(name, df.shape)
    print(df.dtypes)
    print(df.isna().sum(), "\n")

users (278858, 4)
User-ID                        object
Location                       object
Age                            object
ingestion_timestamp    datetime64[us]
dtype: object
User-ID                     0
Location                    0
Age                    110762
ingestion_timestamp         0
dtype: int64 

books (271360, 9)
ISBN                           object
Book-Title                     object
Book-Author                    object
Year-Of-Publication            object
Publisher                      object
Image-URL-S                    object
Image-URL-M                    object
Image-URL-L                    object
ingestion_timestamp    datetime64[us]
dtype: object
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
ingestion_timestamp    0
dtype: int64 

ratings (1149780, 4)
User-ID                        object
ISBN    

In [17]:
# 2.2 Rating distribution
ratings["Book-Rating"] = pd.to_numeric(ratings["Book-Rating"], errors="coerce")
print(ratings["Book-Rating"].value_counts().sort_index())
print("Zeros:", (ratings["Book-Rating"] == 0).sum())

Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64
Zeros: 716109


In [18]:
# 2.3 Age and publication year
users["Age"] = pd.to_numeric(users["Age"], errors="coerce")
print(users["Age"].describe())
print("Age <5:", (users["Age"] < 5).sum(), " Age >100:", (users["Age"] > 100).sum())

books["Year-Of-Publication"] = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
print(books["Year-Of-Publication"].describe())
print("Year 0:", (books["Year-Of-Publication"] == 0).sum(),
      " Future:", (books["Year-Of-Publication"] > 2026).sum())

count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
25%          24.000000
50%          32.000000
75%          44.000000
max         244.000000
Name: Age, dtype: float64
Age <5: 882  Age >100: 366
count    271357.000000
mean       1959.760817
std         257.994226
min           0.000000
25%        1989.000000
50%        1995.000000
75%        2000.000000
max        2050.000000
Name: Year-Of-Publication, dtype: float64
Year 0: 4618  Future: 11


In [19]:
# 2.4 Ratings per user and per book
per_user = ratings.groupby("User-ID").size()
per_book = ratings.groupby("ISBN").size()
print(per_user.describe()); print(per_book.describe())
print("Users with <10 ratings:", (per_user < 10).mean())
print("Books with <10 ratings:", (per_book < 10).mean())

count    105283.000000
mean         10.920851
std          90.562825
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max       13602.000000
dtype: float64
count    340556.000000
mean          3.376185
std          12.436252
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        2502.000000
dtype: float64
Users with <10 ratings: 0.8756019490326074
Books with <10 ratings: 0.9462085530720351


In [20]:
# 2.5 Sparsity
n_users = ratings["User-ID"].nunique()
n_books = ratings["ISBN"].nunique()
sparsity = 100 * (1 - len(ratings) / (n_users * n_books))
print(f"Sparsity: {sparsity:.4f}%")

Sparsity: 99.9968%


In [21]:
# Convert types
ratings["User-ID"] = ratings["User-ID"].astype(int)
users["User-ID"]   = users["User-ID"].astype(int)

# 1. Remove duplicates
users   = users.drop_duplicates()
books   = books.drop_duplicates(subset="ISBN")
ratings = ratings.drop_duplicates(subset=["User-ID", "ISBN"])

# 2. Keep ratings 1-10
ratings = ratings[ratings["Book-Rating"].between(1, 10)]

# 3. Fix ages
users.loc[(users["Age"] < 5) | (users["Age"] > 100), "Age"] = np.nan
users["Age"] = users["Age"].fillna(users["Age"].median())

# 4. Split location into city, state, country
loc = users["Location"].str.split(",", expand=True)
users["city"]    = loc[0].str.strip()
users["state"]   = loc[1].str.strip() if loc.shape[1] > 1 else None
users["country"] = loc[loc.shape[1]-1].str.strip()
users = users.drop(columns="Location")

# 5. Fix publication year
books.loc[(books["Year-Of-Publication"] == 0) |
          (books["Year-Of-Publication"] > 2026), "Year-Of-Publication"] = np.nan

# 6. Clean text, drop image columns
books = books.drop(columns=[c for c in books.columns if c.startswith("Image-URL")])
for col in ["Book-Title", "Book-Author", "Publisher"]:
    books[col] = books[col].str.strip().str.replace(r"\s+", " ", regex=True).str.title()

# Fix apostrophe capitalization caused by .title() (e.g. "Sorcerer'S" -> "Sorcerer's")
for col in ["Book-Title", "Book-Author", "Publisher"]:
    books[col] = books[col].str.replace(r"'S\b", "'s", regex=True)

for col in ["city", "state", "country"]:
    users[col] = users[col].str.strip().str.lower()

# 7. Referential integrity
ratings = ratings[ratings["ISBN"].isin(books["ISBN"]) &
                  ratings["User-ID"].isin(users["User-ID"])]

# 8. Minimum-ratings threshold (choose and justify!)
MIN_USER, MIN_BOOK = 10, 10
for _ in range(3):  # repeat because filtering one side affects the other
    ratings = ratings[ratings.groupby("User-ID")["ISBN"].transform("count") >= MIN_USER]
    ratings = ratings[ratings.groupby("ISBN")["User-ID"].transform("count") >= MIN_BOOK]

users = users[users["User-ID"].isin(ratings["User-ID"])]
books = books[books["ISBN"].isin(ratings["ISBN"])]

print(users.shape, books.shape, ratings.shape)

users.to_parquet("silver/users.parquet", index=False)
books.to_parquet("silver/books.parquet", index=False)
ratings.to_parquet("silver/ratings.parquet", index=False)

(1896, 6) (2077, 6) (42548, 4)


In [22]:
import scipy.sparse as sp
import pickle

users   = pd.read_parquet("silver/users.parquet")
books   = pd.read_parquet("silver/books.parquet")
ratings = pd.read_parquet("silver/ratings.parquet")

# Map ids to matrix indices
user_ids = ratings["User-ID"].astype("category")
isbn_ids = ratings["ISBN"].astype("category")

matrix = sp.csr_matrix(
    (ratings["Book-Rating"].values,
     (user_ids.cat.codes, isbn_ids.cat.codes))
)  # rows = users, columns = books

sp.save_npz("gold/user_book_matrix.npz", matrix)
pickle.dump({"users": list(user_ids.cat.categories),
             "isbns": list(isbn_ids.cat.categories)},
            open("gold/index_maps.pkl", "wb"))

# Book lookup
lookup = books[["ISBN", "Book-Title", "Book-Author"]]
lookup.to_parquet("gold/book_lookup.parquet", index=False)

# Aggregates
user_agg = ratings.groupby("User-ID")["Book-Rating"].agg(
    avg_rating="mean", num_ratings="count").reset_index()
book_agg = ratings.groupby("ISBN")["Book-Rating"].agg(
    avg_rating="mean", num_ratings="count").reset_index()
user_agg.to_parquet("gold/user_aggregates.parquet", index=False)
book_agg.to_parquet("gold/book_aggregates.parquet", index=False)

In [23]:
from sklearn.neighbors import NearestNeighbors

# 5.1 Load and transpose: rows = books, columns = users
matrix = sp.load_npz("gold/user_book_matrix.npz")
maps = pickle.load(open("gold/index_maps.pkl", "rb"))
lookup = pd.read_parquet("gold/book_lookup.parquet")

book_user = matrix.T.tocsr()
isbns, uids = maps["isbns"], maps["users"]
isbn_to_idx = {i: k for k, i in enumerate(isbns)}
uid_to_idx = {u: k for k, u in enumerate(uids)}
isbn_to_title = dict(zip(lookup["ISBN"], lookup["Book-Title"]))

# 5.2 Cosine similarity with NearestNeighbors
nn = NearestNeighbors(metric="cosine", algorithm="brute")
nn.fit(book_user)

NearestNeighbors(algorithm='brute', metric='cosine')

In [24]:
# 5.3 Similar books by title
def similar_books(title, n=10):
    matches = lookup[lookup["Book-Title"].str.lower() == title.lower()]
    if matches.empty:
        return f"'{title}' not found"
    idx = isbn_to_idx[matches.iloc[0]["ISBN"]]
    dist, ind = nn.kneighbors(book_user[idx], n_neighbors=n + 1)
    return [(isbn_to_title[isbns[i]], round(1 - d, 3))
            for d, i in zip(dist[0][1:], ind[0][1:])]

In [25]:
def recommend(user_id, n=10, min_rating=7):
    row = matrix[uid_to_idx[user_id]]
    rated = dict(zip(row.indices, row.data))
    rated_titles = {isbn_to_title[isbns[i]].lower() for i in rated}
    liked = {i: r for i, r in rated.items() if r >= min_rating}

    scores = {}
    for book_idx, rating in liked.items():
        dist, ind = nn.kneighbors(book_user[book_idx], n_neighbors=11)
        for d, j in zip(dist[0][1:], ind[0][1:]):
            if j in rated:
                continue
            scores[j] = scores.get(j, 0) + (1 - d) * rating

    results, seen = [], set()
    for j, _ in sorted(scores.items(), key=lambda x: -x[1]):
        title = isbn_to_title[isbns[j]]
        key = title.lower()
        if key in rated_titles or key in seen:
            continue
        seen.add(key)
        results.append(title)
        if len(results) == n:
            break
    return results

In [26]:
sample_users = pd.Series(uids).sample(3, random_state=42).tolist()

for u in sample_users:
    row = matrix[uid_to_idx[u]]
    liked = [(isbn_to_title[isbns[i]], r) for i, r in zip(row.indices, row.data) if r >= 8]
    print(f"\n=== User {u} ===")
    print("Highly rated books:")
    for t, r in liked[:8]: print(f"  {r}: {t}")
    print("Recommendations:")
    for t in recommend(u): print("  -", t)


=== User 194287 ===
Highly rated books:
  8: The Queen Of The Damned (Vampire Chronicles (Paperback))
  10: Glory In Death
  10: Dark Magic
  10: Dark Challenge
  10: Blue Moon
  10: Bloody Bones (Anita Blake Vampire Hunter (Paperback))
  10: Circus Of The Damned (Anita Blake Vampire Hunter (Paperback))
  10: Guilty Pleasures (Anita Blake Vampire Hunter (Paperback))
Recommendations:
  - The Laughing Corpse (Anita Blake Vampire Hunter (Paperback))
  - Narcissus In Chains (Anita Blake Vampire Hunter (Paperback))
  - Dark Gold
  - Dark Desire (Paranormal Romance)
  - Cerulean Sins: An Anita Blake, Vampire Hunter Novel
  - Dark Symphony
  - Dark Guardian
  - Lair Of The Lion
  - After Twilight
  - When The Wind Blows

=== User 92810 ===
Highly rated books:
  8: Moment Of Truth
  9: The Cider House Rules
  8: Protect And Defend
  8: A Widow For One Year
  9: The Emperor Of Ocean Park (Today Show Book Club #1)
  9: The Da Vinci Code
  8: Sick Puppy
  8: Saving Faith
Recommendations:
  - The

# Data Mining Lab: Book Recommender using a Medallion Data Lake

**Dataset:** Book-Crossing (Users, Books, Book-Ratings)
**Pipeline:** Bronze (raw) → Silver (cleaned) → Gold (model-ready) → Item-based Collaborative Filtering

### 2.1 Shape, data types and missing values

- **Users:** 278,858 rows. `Age` is missing for 110,762 users (about 39.7%). All columns are strings because Bronze is the raw copy. **Silver:** convert `User-ID` and `Age` to numeric, set impossible ages to null, and fill nulls with the median age.
- **Books:** 271,360 rows. Very few missing values (2 in `Book-Author`, 2 in `Publisher`, 3 in `Image-URL-L`). **Silver:** drop the image-URL columns and convert `Year-Of-Publication` to numeric.
- **Ratings:** 1,149,780 rows with no missing values. **Silver:** convert `User-ID` and `Book-Rating` to integers.

### 2.2 Rating distribution

**Found:** 716,109 of 1,149,780 ratings (about 62.3%) are 0, which means "no explicit rating". Only 433,671 (about 37.7%) are real ratings from 1 to 10, and they lean high (8 is the most common).
**Silver:** remove all 0 ratings and keep only ratings from 1 to 10.

### 2.3 User ages and publication years

**Found (age):** 882 ages are below 5 and 366 are above 100 (minimum 0, maximum 244). The median of the known ages is 32.
**Silver:** set ages below 5 or above 100 to null, then fill all nulls with the median.

**Found (year):** 4,618 books have year 0 and 11 have a future year (maximum 2050). The median year is 1995.
**Silver:** set year 0 and years after 2026 to null.

### 2.4 Ratings per user and per book

**Found:** The median is 1 rating per user and 1 per book. 87.6% of users and 94.6% of books have fewer than 10 ratings. A few power users and books dominate (up to 13,602 ratings for one user and 2,502 for one book). Also, the ratings contain more distinct ISBNs (340,556) than the books table has rows (271,360), so some ratings point to books that do not exist in the books table.
**Silver:** keep only ratings whose user and book exist in the users and books tables, then apply a minimum-ratings threshold.

### 2.5 Matrix sparsity

**Found:** The user-book matrix is 99.9968% empty. Only about 0.003% of possible user-book pairs have a rating.
**Silver/Gold:** filter out low-activity users and books to reduce noise, and store the matrix in Gold as a sparse matrix (CSR).

### Silver threshold: at least 10 ratings per user and per book

The EDA showed that the median user and the median book each have only 1 rating, and 87.6% of users and 94.6% of books have fewer than 10 ratings. Similarity computed from only 1 or 2 ratings is unreliable, because two books rated by the same single user look "perfectly similar" by chance. A user with only a few ratings also gives too little signal about their taste.

I chose 10 as a balance: a lower value leaves too much noise, and a higher value would discard most of the data. The filter is applied in a loop, because removing users changes the book counts and the other way around.

**Result:** Silver has 1,896 users, 2,077 books and 42,548 ratings. About 1.1% of the matrix cells are now filled, compared with 0.003% before. This is a large reduction (only about 10% of explicit ratings are kept), but it makes item-based similarity much more reliable.

### Sanity check

- **User 194287** rated the *Anita Blake* series and other paranormal fiction highly. The recommendations are other *Anita Blake* books (*The Laughing Corpse*, *Narcissus in Chains*, *Cerulean Sins*) plus similar paranormal romance (*Dark Gold*, *Dark Desire*). Very strong match.
- **User 92810** rated legal thrillers and mysteries highly (*Protect and Defend*, *Saving Faith*, *The Da Vinci Code*). The recommendations are mostly thrillers and mysteries (*The Simple Truth*, *Unfit to Practice*, *The Empty Chair*). A few, like *A Walk to Remember*, are less related.
- **User 141511** rated *Harry Potter and the Order of the Phoenix* highly, and the recommender suggests the other Harry Potter books, as expected.

**Overall:** the recommendations are clearly related to what each user liked, usually by the same series, author or genre. Books the user has already read are excluded, and duplicate titles are removed.

**Limitations:**
1. Editions with slightly different titles (e.g. "Sorcerer's Stone (Book 1)" and "(Harry Potter (Paperback))") can still appear twice.
2. Item-based CF uses only rating patterns, so some recommendations are popular books rather than close matches.
3. Users or books with few ratings cannot be recommended well (cold-start problem).